# Gaussian Elimination: Solving Systems Step by Step

**This notebook teaches Gaussian elimination from scratch, it assumes only that you can add and scale equations.**

The previous notebook solved a small system by elimination. **Gaussian elimination** is that same idea made systematic: instead of juggling equations, we write the system as a grid of numbers, the **augmented matrix**, and apply a few simple **row operations** until the answer can be read off. We work a 2×2 first, then a 3×3, showing every step; then we let the computer confirm it.

## Learning objectives

- Write a system as an **augmented matrix**.
- Apply the three **row operations** to reach **row-echelon form**.
- Recover the solution by **back-substitution**, first on a 2×2 then a 3×3, and check with `numpy`.

## Background

A system of equations is bookkeeping: the same variables appear in every equation, and only the **coefficients** change. Strip away the letters and plus signs and what is left is a grid of numbers, the coefficients, with the right-hand sides attached in a final column, separated by a bar. That grid is the **augmented matrix**. For

$$\begin{aligned} x + y &= 3\\ 2x + y &= 5 \end{aligned} \qquad\text{we write}\qquad \left[\begin{array}{cc|c} 1 & 1 & 3\\ 2 & 1 & 5 \end{array}\right]$$

Three **row operations** never change the solution:

1. swap two rows,
2. multiply a row by a nonzero number,
3. add a multiple of one row to another.

Using these to create zeros below the diagonal gives **row-echelon form**; then **back-substitution** (solve the bottom row, work upward) yields the answer. Each augmented matrix below is displayed with a vertical bar separating the coefficients from the right-hand side.

## This notebook covers

1. The augmented matrix and the row operations.
2. A worked 2×2 example.
3. A worked 3×3 example.
4. Back-substitution.
5. Letting the computer solve it.

**Prerequisites:** `U2-1_Systems-2_SubstitutionAndElimination`

**Dataset:** none, the systems are entered directly.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# shared display helper (msds520_helpers.py lives at the MSDS 520 course root).
# notebooks sit two folders below it, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds520_helpers as helpers

## 1. The augmented matrix and the row operations

Elimination on equations is exactly **row operations** on the augmented matrix. The three moves, **swap** two rows, **scale** a row by a nonzero number, and **add a multiple** of one row to another, leave the solution unchanged, because each one is something you could legitimately do to the equations themselves. We will use scaling and adding to create zeros below the diagonal.

## 2. A worked 2×2 example

Solve $x + y = 3$ and $2x + y = 5$. We build $[A \mid \mathbf{b}]$ and clear the entry below the first pivot.

In [2]:
# Augmented matrix for  x + y = 3,  2x + y = 5
M = np.array([[1.0, 1, 3],
              [2.0, 1, 5]])
helpers.show_aug(M[:, :-1], M[:, -1], label="start")

<IPython.core.display.Math object>

In [3]:
# Clear the entry below the first pivot:  R2 -> R2 - 2*R1
M[1] = M[1] - 2 * M[0]
helpers.show_aug(M[:, :-1], M[:, -1], label=r"R_2 \to R_2 - 2R_1")

<IPython.core.display.Math object>

In [4]:
# Make the second pivot equal to 1:  R2 -> -1 * R2
M[1] = -1 * M[1]
helpers.show_aug(M[:, :-1], M[:, -1], label=r"R_2 \to -R_2 (row-echelon form)")

<IPython.core.display.Math object>

The bottom row now says $y = 1$. Back-substitute into the top row $x + y = 3$ to get $x = 2$. So $(x, y) = (2, 1)$.

In [5]:
y = M[1, 2]                      # bottom row: y = ...
x = M[0, 2] - M[0, 1] * y        # top row: x + (coef)*y = rhs
print("x =", x, " y =", y)

x = 2.0  y = 1.0


## 3. A worked 3×3 example

The same procedure scales up. Solve

$$2x + 4y + 4z = 4, \qquad x + 3y + z = 4, \qquad -x + 3y + 2z = -1.$$

We clear the first column, then the second, printing the augmented matrix after each row operation.

In [6]:
M = np.array([[ 2.0, 4, 4,  4],
              [ 1.0, 3, 1,  4],
              [-1.0, 3, 2, -1]])
helpers.show_aug(M[:, :-1], M[:, -1], label="start")

<IPython.core.display.Math object>

In [7]:
# Step 1: make the top-left pivot 1  ->  R1 = R1 / 2
M[0] = M[0] / 2
helpers.show_aug(M[:, :-1], M[:, -1], label=r"R_1 \to R_1 / 2")

<IPython.core.display.Math object>

In [8]:
# Step 2: clear the rest of column 1  ->  R2 = R2 - R1,  R3 = R3 + R1
M[1] = M[1] - M[0]
M[2] = M[2] + M[0]
helpers.show_aug(M[:, :-1], M[:, -1], label=r"R_2 \to R_2 - R_1, \; R_3 \to R_3 + R_1")

<IPython.core.display.Math object>

In [9]:
# Step 3: clear below the second pivot  ->  R3 = R3 - 5*R2
M[2] = M[2] - 5 * M[1]
helpers.show_aug(M[:, :-1], M[:, -1], label=r"R_3 \to R_3 - 5R_2")

<IPython.core.display.Math object>

In [10]:
# Step 4: make the last pivot 1  ->  R3 = R3 / 9
M[2] = M[2] / 9
helpers.show_aug(M[:, :-1], M[:, -1], label=r"R_3 \to R_3 / 9 (row-echelon form)")

<IPython.core.display.Math object>

## 4. Back-substitution

The bottom row gives $z = -1$. The middle row is $y - z = 2$, so $y = 2 + z = 1$; the top row is $x + 2y + 2z = 2$, so $x = 2 - 2y - 2z = 2$. The solution is $(x, y, z) = (2, 1, -1)$.

In [11]:
z = M[2, 3]                              # bottom row
y = M[1, 3] - M[1, 2] * z                # middle row: y - (coef)*z = rhs
x = M[0, 3] - M[0, 1] * y - M[0, 2] * z  # top row
print("x =", x, " y =", y, " z =", z)
helpers.show(np.array([x, y, z]), name="solution")

x = 2.0  y = 1.0  z = -1.0


<IPython.core.display.Math object>

## 5. Letting the computer solve it

`np.linalg.solve` runs this same elimination internally. Hand it the coefficient grid and the right-hand side and it returns the solution. We confirm both systems worked above.

In [12]:
# the coefficients and right-hand sides of the two systems worked above
coeffs2 = np.array([[1.0, 1], [2.0, 1]]);              rhs2 = np.array([3.0, 5])
coeffs3 = np.array([[2.0, 4, 4], [1.0, 3, 1], [-1.0, 3, 2]]);  rhs3 = np.array([4.0, 4, -1])

helpers.show(np.linalg.solve(coeffs2, rhs2), label=r"2 \times 2 solution")
helpers.show(np.linalg.solve(coeffs3, rhs3), label=r"3 \times 3 solution")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 6. Summary

- A system becomes an **augmented matrix**, coefficients, with the right-hand sides in a final column; three **row operations** simplify it without changing the solution.
- Clearing below each pivot gives **row-echelon form**; **back-substitution** reads off the answer.
- The method is identical for a 2×2 and a 3×3, just more steps.
- `np.linalg.solve` does the whole thing for you.

Next: vectors, the other half of linear algebra, and the objects these systems are really about (`U2-2_Vectors-1_ArrowsAndOps`).